In [1]:
from google.colab import drive
import tarfile
import os
from os.path import join as pjoin

# 구글 드라이브 연결
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install monai
!pip install nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.2 MB/s eta 0:00:00


In [3]:
from monai.data import ImageDataset
from monai.transforms import (
    EnsureChannelFirst,
    ToTensor,
 )
import torch
from torch.utils.data import DataLoader

from monai.transforms import Compose
from monai.data import DataLoader, Dataset
from glob import glob

# 데이터셋 경로 설정
data_dir = '/content/drive/MyDrive/IXI_Dataset'
img_dir = pjoin(data_dir, 'images')

# 전체 이미지 리스트
img_list = glob(pjoin(img_dir, '*.nii.gz'))

sample_ds = ImageDataset(image_files=img_list, transform=Compose([
    EnsureChannelFirst(),
    ToTensor(),
]))

sample_loader = DataLoader(sample_ds, batch_size=1)

# 데이터 확인
first_sample = next(iter(sample_loader))

print(first_sample.shape)
print(first_sample.min())
print(first_sample.max())
print(torch.isnan(first_sample).any())
print(first_sample)

torch.Size([1, 1, 256, 256, 130])
metatensor(0.)
metatensor(3262.7561)
metatensor(False)
metatensor([[[[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           ...,
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

          [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           ...,
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000

In [5]:
import os
import sys

import matplotlib.pyplot as plt
import torch
import numpy as np

import monai
from monai.config import print_config
from monai.data import DataLoader, ImageDataset
from monai.transforms import (
    EnsureChannelFirst,
    Compose,
    RandRotate90,
    Resize,
    ScaleIntensity,
)
import pandas as pd
from glob import glob

pin_memory = torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print_config()

MONAI version: 1.4.0
Numpy version: 1.26.4
Pytorch version: 2.5.0+cu121
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 46a5272196a6c2590ca2589029eed8e4d56ff008
MONAI __file__: /usr/local/lib/python3.10/dist-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.2.1
scikit-image version: 0.24.0
scipy version: 1.13.1
Pillow version: 10.4.0
Tensorboard version: 2.17.0
gdown version: 5.2.0
TorchVision version: 0.20.0+cu121
tqdm version: 4.66.5
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 5.9.5
pandas version: 2.2.2
einops version: 0.8.0
transformers version: 4.44.2
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/l

In [ ]:
# Define transforms
train_transforms = Compose([
    ScaleIntensity(),
    EnsureChannelFirst(),
    Resize((96, 96, 96)),
    RandRotate90()
])

val_transforms = Compose([
    ScaleIntensity(),
    EnsureChannelFirst(),
    Resize((96, 96, 96))
])

# declare images, labels
img_path_list =  glob(os.path.join(img_dir, '*.nii.gz'))
meta_data = pd.read_excel(os.path.join(data_dir, 'IXI.xls'))

# random split dataset
val_data_length = int(len(img_path_list) * 0.2)
train_data_length = len(img_path_list) - val_data_length
tr_img_path_list, val_img_path_list = torch.utils.data.random_split(img_path_list, [train_data_length, val_data_length])
ts_data_length = len(val_img_path_list) // 2
ts_img_path_list, val_img_path_list = torch.utils.data.random_split(val_img_path_list, [ts_data_length, len(val_img_path_list) - ts_data_length])


In [ ]:
def get_label(img_path_list):
    images = []
    labels = []
    for img_path in img_path_list:
        img_id = int(img_path.split('/')[-1][3:6])
        sex_value = meta_data[meta_data['IXI_ID'] == img_id]["SEX_ID (1=m, 2=f)"].values
        if len(sex_value) > 0:
            images.append(img_path)
            labels.append(sex_value[0] - 1) # m = 0, f = 1
    labels = torch.nn.functional.one_hot(torch.as_tensor(labels)).float()
    return images, labels

tr_images, tr_labels = get_label(tr_img_path_list)
val_images, val_labels = get_label(val_img_path_list)
ts_images, ts_labels = get_label(ts_img_path_list)

print(len(tr_images), len(tr_labels))
print(tr_images[:5])
print(tr_labels[:5])

In [ ]:
# Define nifti dataset, data loader
check_ds = ImageDataset(image_files=tr_images, labels=tr_labels, transform=train_transforms)
check_loader = DataLoader(check_ds, batch_size=3, num_workers=2, pin_memory=pin_memory)
im, label = monai.utils.misc.first(check_loader)
print(type(im), im.shape, label, label.shape)

# create a training data loader
train_ds = ImageDataset(image_files=tr_images, labels=tr_labels, transform=train_transforms)
 train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2,
pin_memory=pin_memory)
 # create a validation data loader
 val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)
 val_loader = DataLoader(val_ds, batch_size=32, num_workers=2, pin_memory=pin_memory)
 # create a validation data loader
 ts_ds = ImageDataset(image_files=ts_images, labels=ts_labels, transform=val_transforms)
 ts_loader = DataLoader(val_ds, batch_size=1, num_workers=2, pin_memory=pin_memory)

 # Create DenseNet121, CrossEntropyLoss and Adam optimizer
 model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1,
out_channels=2).to(device)
 loss_function = torch.nn.CrossEntropyLoss()
 optimizer = torch.optim.Adam(model.parameters(), 1e-4)
 # start a typical PyTorch training
 val_interval = 1
 best_metric = -1
 best_metric_epoch = -1
 epoch_loss_values = []
 metric_values = []
 max_epochs = 5
 for epoch in range(max_epochs):
 print("-" * 10)
 print(f"epoch {epoch + 1}/{max_epochs}")
 model.train()
 epoch_loss = 0
 step = 0
 for batch_data in train_loader:
 step += 1
 inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
 optimizer.zero_grad()
 outputs = model(inputs)
 loss = loss_function(outputs, labels)
 loss.backward()
 optimizer.step()
 epoch_loss += loss.item()
 epoch_len = len(train_ds) // train_loader.batch_size
 print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
 epoch_loss /= step
 epoch_loss_values.append(epoch_loss)
 print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
 if (epoch + 1) % val_interval == 0:
 model.eval()
 num_correct = 0.0
 metric_count = 0
 for val_data in val_loader:
 val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
  with torch.no_grad():
 val_outputs = model(val_images)
 value = torch.eq(val_outputs.argmax(dim=1), val_labels.argmax(dim=1))
 metric_count += len(value)
 num_correct += value.sum().item()
 metric = num_correct / metric_count
 metric_values.append(metric)
  if metric > best_metric:
  best_metric = metric
  best_metric_epoch = epoch + 1
  torch.save(model.state_dict(), "best_metric_model_classification3d_array.pth") #
torch.load
  print("saved new best metric model")
 print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
 print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
 print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")